In [2]:
# ============================================================
# Cell 0: Install required packages in the active notebook kernel
# ============================================================

import sys

!{sys.executable} -m pip install pandas requests python-dotenv pyyaml

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# ============================================================
# Cell 1: Imports and project root setup
# ============================================================

from pathlib import Path
import os
import json
import requests

import pandas as pd
import yaml
from dotenv import load_dotenv

# Notebook is inside: use_cases/meteoblue/notebooks
# Project root should be: use_cases/meteoblue
PROJECT_ROOT = Path.cwd().parent

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())

Current working directory: c:\Users\dubey\Documents\UC_GitLab\use_cases\meteoblue\notebooks
Project root: c:\Users\dubey\Documents\UC_GitLab\use_cases\meteoblue
Project root exists: True


In [2]:
# ============================================================
# Cell 2: Load API key, config file, and input locations
# ============================================================

# 1. Load API key from .env
env_path = PROJECT_ROOT / ".env"
load_dotenv(env_path)

METEOBLUE_API_KEY = os.getenv("METEOBLUE_API_KEY")

if not METEOBLUE_API_KEY:
    raise ValueError("METEOBLUE_API_KEY was not found. Please check your .env file.")

print("API key loaded successfully.")
print("API key length:", len(METEOBLUE_API_KEY))

# 2. Load YAML config
config_path = PROJECT_ROOT / "config" / "meteoblue_config.yaml"

with open(config_path, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

print("\nConfig loaded successfully:")
print(config)

# 3. Load input CSV
input_file = PROJECT_ROOT / config["input"]["file_path"]
locations_df = pd.read_csv(input_file)

print("\nLocations loaded successfully:")
display(locations_df)

API key loaded successfully.
API key length: 16

Config loaded successfully:
{'input': {'file_path': 'data/input/sample_location.csv'}, 'output': {'raw_folder': 'data/raw', 'output_folder': 'data/output'}, 'api': {'timeout_seconds': 60}}

Locations loaded successfully:


,location_id,latitude,longitude,country_code,start_date,end_date
0,LOC001,34.12345,-78.23456,US,2024-04-10,2024-04-15


In [3]:
# ============================================================
# Cell 3: Select one sample location from the input file
# ============================================================

# For now, we use the first row.
# Later, this same code can be placed inside a loop for many locations.
sample_location = locations_df.iloc[0].to_dict()

location_id = sample_location["location_id"]
latitude = float(sample_location["latitude"])
longitude = float(sample_location["longitude"])
country_code = sample_location["country_code"]
start_date = sample_location["start_date"]
end_date = sample_location["end_date"]

print("Selected location:")
print("location_id:", location_id)
print("latitude:", latitude)
print("longitude:", longitude)
print("country_code:", country_code)
print("start_date:", start_date)
print("end_date:", end_date)

Selected location:
location_id: LOC001
latitude: 34.12345
longitude: -78.23456
country_code: US
start_date: 2024-04-10
end_date: 2024-04-15


In [4]:
# ============================================================
# Cell 4: Test Meteoblue basic-day API call
# ============================================================

basic_day_url = "https://my.meteoblue.com/packages/basic-day"

params = {
    "apikey": METEOBLUE_API_KEY,
    "lat": latitude,
    "lon": longitude,
    "asl": 0,
    "format": "json",
}

response = requests.get(
    basic_day_url,
    params=params,
    timeout=config["api"]["timeout_seconds"]
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))

# Show first 500 characters without printing too much
print("\nFirst 500 characters of response:")
print(response.text[:500])

# Raise an error if the request failed
response.raise_for_status()

basic_day_json = response.json()

print("\nTop-level keys:")
print(list(basic_day_json.keys()))

Status code: 200
Content type: application/json; charset=utf-8

First 500 characters of response:
{"metadata":{"modelrun_updatetime_utc":"2026-05-11 03:23","name":"","height":0,"timezone_abbrevation":"EDT","latitude":34.12345,"modelrun_utc":"2026-05-11 03:23","longitude":-78.23456,"utc_timeoffset":-4.0,"generation_time_ms":8.998036},"units":{"predictability":"percent","precipitation":"mm","windspeed":"ms-1","precipitation_probability":"percent","relativehumidity":"percent","time":"YYYY-MM-DD hh:mm","temperature":"C","pressure":"hPa","winddirection":"degree"},"data_day":{"time":["2026-05-10",

Top-level keys:
['metadata', 'units', 'data_day']


In [5]:
# ============================================================
# Cell 5: Save raw basic-day weather JSON response
# ============================================================

raw_folder = PROJECT_ROOT / config["output"]["raw_folder"]
raw_folder.mkdir(parents=True, exist_ok=True)

raw_weather_file = raw_folder / f"basic_day_{location_id}.json"

with open(raw_weather_file, "w", encoding="utf-8") as file:
    json.dump(basic_day_json, file, indent=2)

print("Raw weather JSON saved successfully.")
print("File path:", raw_weather_file)
print("File exists:", raw_weather_file.exists())

Raw weather JSON saved successfully.
File path: c:\Users\dubey\Documents\UC_GitLab\use_cases\meteoblue\data\raw\basic_day_LOC001.json
File exists: True


In [6]:
# ============================================================
# Cell 6: Inspect basic-day weather JSON structure
# ============================================================

print("Top-level keys:")
print(list(basic_day_json.keys()))

print("\nMetadata:")
for key, value in basic_day_json.get("metadata", {}).items():
    print(f"{key}: {value}")

print("\nUnits:")
for key, value in basic_day_json.get("units", {}).items():
    print(f"{key}: {value}")

print("\nData day keys:")
data_day = basic_day_json.get("data_day", {})
print(list(data_day.keys()))

print("\nNumber of forecast days:")
print(len(data_day.get("time", [])))

print("\nFirst 3 dates:")
print(data_day.get("time", [])[:3])

Top-level keys:
['metadata', 'units', 'data_day']

Metadata:
modelrun_updatetime_utc: 2026-05-11 03:23
name: 
height: 0
timezone_abbrevation: EDT
latitude: 34.12345
modelrun_utc: 2026-05-11 03:23
longitude: -78.23456
utc_timeoffset: -4.0
generation_time_ms: 8.998036

Units:
predictability: percent
precipitation: mm
windspeed: ms-1
precipitation_probability: percent
relativehumidity: percent
time: YYYY-MM-DD hh:mm
temperature: C
pressure: hPa
winddirection: degree

Data day keys:
['time', 'temperature_instant', 'precipitation', 'predictability', 'temperature_max', 'sealevelpressure_mean', 'windspeed_mean', 'precipitation_hours', 'sealevelpressure_min', 'pictocode', 'snowfraction', 'humiditygreater90_hours', 'convective_precipitation', 'relativehumidity_max', 'temperature_min', 'winddirection', 'felttemperature_max', 'relativehumidity_min', 'felttemperature_mean', 'windspeed_min', 'felttemperature_min', 'precipitation_probability', 'uvindex', 'rainspot', 'temperature_mean', 'sealevelpres

In [7]:
# ============================================================
# Cell 7: Convert basic-day weather JSON to pandas DataFrame
# ============================================================

data_day = basic_day_json["data_day"]

# Convert dictionary of equal-length arrays into a dataframe
weather_daily_df = pd.DataFrame(data_day)

# Add location metadata
weather_daily_df.insert(0, "location_id", location_id)
weather_daily_df.insert(1, "latitude", latitude)
weather_daily_df.insert(2, "longitude", longitude)
weather_daily_df.insert(3, "country_code", country_code)

# Convert time column to date
weather_daily_df["time"] = pd.to_datetime(weather_daily_df["time"]).dt.date

# Rename time to weather_date for clearer data modeling
weather_daily_df = weather_daily_df.rename(columns={"time": "weather_date"})

print("Weather daily dataframe shape:", weather_daily_df.shape)
display(weather_daily_df.head())

Weather daily dataframe shape: (7, 33)


,location_id,latitude,longitude,country_code,weather_date,temperature_instant,precipitation,predictability,temperature_max,sealevelpressure_mean,...,windspeed_min,felttemperature_min,precipitation_probability,uvindex,rainspot,temperature_mean,sealevelpressure_max,relativehumidity_mean,predictability_class,windspeed_max
0,LOC001,34.12345,-78.23456,US,2026-05-10,17.07,0.00,87,29.31,1014,...,0.72,16.87,0,7,0000000000000000000000000000000000000000000000000,21.72,1016,81,5.0,2.44
1,LOC001,34.12345,-78.23456,US,2026-05-11,17.94,9.86,55,28.33,1016,...,0.56,12.43,96,7,3333222333332233333322223333222223322222222222222,20.80,1019,86,3.0,2.71
2,LOC001,34.12345,-78.23456,US,2026-05-12,13.19,0.00,70,23.37,1021,...,0.88,9.56,0,8,1111101119000001000000000000000000019910991111990,16.47,1022,65,4.0,3.00
3,LOC001,34.12345,-78.23456,US,2026-05-13,11.81,0.00,48,24.99,1017,...,0.61,10.32,84,5,0000000000000000000000000000990000011190000000000,18.79,1021,71,3.0,3.77
4,LOC001,34.12345,-78.23456,US,2026-05-14,16.66,11.12,46,26.03,1011,...,1.15,13.28,47,6,3333333333333333333333333333333333333333333333333,20.34,1014,73,3.0,2.87


In [8]:
# ============================================================
# Cell 8: Save wide weather DataFrame to output folder
# ============================================================

output_folder = PROJECT_ROOT / config["output"]["output_folder"]
output_folder.mkdir(parents=True, exist_ok=True)

weather_wide_file = output_folder / f"weather_basic_day_wide_{location_id}.csv"

weather_daily_df.to_csv(weather_wide_file, index=False)

print("Weather wide CSV saved successfully.")
print("File path:", weather_wide_file)
print("File exists:", weather_wide_file.exists())

Weather wide CSV saved successfully.
File path: c:\Users\dubey\Documents\UC_GitLab\use_cases\meteoblue\data\output\weather_basic_day_wide_LOC001.csv
File exists: True


In [9]:
# ============================================================
# Cell 9: Convert wide weather table to long/fact format
# ============================================================

id_columns = [
    "location_id",
    "latitude",
    "longitude",
    "country_code",
    "weather_date",
]

weather_variable_columns = [
    col for col in weather_daily_df.columns
    if col not in id_columns
]

weather_long_df = weather_daily_df.melt(
    id_vars=id_columns,
    value_vars=weather_variable_columns,
    var_name="weather_variable_name",
    value_name="weather_value"
)

# Add source/context columns
weather_long_df["source_api_package"] = "basic-day"
weather_long_df["extraction_type"] = "FORECAST"
weather_long_df["extraction_ts"] = pd.Timestamp.now()

print("Weather long dataframe shape:", weather_long_df.shape)
display(weather_long_df.head(20))

Weather long dataframe shape: (196, 10)


,location_id,latitude,longitude,country_code,weather_date,weather_variable_name,weather_value,source_api_package,extraction_type,extraction_ts
0,LOC001,34.12345,-78.23456,US,2026-05-10,temperature_instant,17.07,basic-day,FORECAST,2026-05-10 23:41:59.512551
1,LOC001,34.12345,-78.23456,US,2026-05-11,temperature_instant,17.94,basic-day,FORECAST,2026-05-10 23:41:59.512551
2,LOC001,34.12345,-78.23456,US,2026-05-12,temperature_instant,13.19,basic-day,FORECAST,2026-05-10 23:41:59.512551
3,LOC001,34.12345,-78.23456,US,2026-05-13,temperature_instant,11.81,basic-day,FORECAST,2026-05-10 23:41:59.512551
4,LOC001,34.12345,-78.23456,US,2026-05-14,temperature_instant,16.66,basic-day,FORECAST,2026-05-10 23:41:59.512551
5,LOC001,34.12345,-78.23456,US,2026-05-15,temperature_instant,14.14,basic-day,FORECAST,2026-05-10 23:41:59.512551
6,LOC001,34.12345,-78.23456,US,2026-05-16,temperature_instant,13.91,basic-day,FORECAST,2026-05-10 23:41:59.512551
7,LOC001,34.12345,-78.23456,US,2026-05-10,precipitation,0.0,basic-day,FORECAST,2026-05-10 23:41:59.512551
8,LOC001,34.12345,-78.23456,US,2026-05-11,precipitation,9.86,basic-day,FORECAST,2026-05-10 23:41:59.512551
9,LOC001,34.12345,-78.23456,US,2026-05-12,precipitation,0.0,basic-day,FORECAST,2026-05-10 23:41:59.512551


In [10]:
# ============================================================
# Cell 10: Save long weather fact table to output folder
# ============================================================

weather_long_file = output_folder / f"weather_basic_day_long_{location_id}.csv"

weather_long_df.to_csv(weather_long_file, index=False)

print("Weather long/fact CSV saved successfully.")
print("File path:", weather_long_file)
print("File exists:", weather_long_file.exists())

print("\nPreview of saved fact-style weather data:")
display(weather_long_df.head(10))

Weather long/fact CSV saved successfully.
File path: c:\Users\dubey\Documents\UC_GitLab\use_cases\meteoblue\data\output\weather_basic_day_long_LOC001.csv
File exists: True

Preview of saved fact-style weather data:


,location_id,latitude,longitude,country_code,weather_date,weather_variable_name,weather_value,source_api_package,extraction_type,extraction_ts
0,LOC001,34.12345,-78.23456,US,2026-05-10,temperature_instant,17.07,basic-day,FORECAST,2026-05-10 23:41:59.512551
1,LOC001,34.12345,-78.23456,US,2026-05-11,temperature_instant,17.94,basic-day,FORECAST,2026-05-10 23:41:59.512551
2,LOC001,34.12345,-78.23456,US,2026-05-12,temperature_instant,13.19,basic-day,FORECAST,2026-05-10 23:41:59.512551
3,LOC001,34.12345,-78.23456,US,2026-05-13,temperature_instant,11.81,basic-day,FORECAST,2026-05-10 23:41:59.512551
4,LOC001,34.12345,-78.23456,US,2026-05-14,temperature_instant,16.66,basic-day,FORECAST,2026-05-10 23:41:59.512551
5,LOC001,34.12345,-78.23456,US,2026-05-15,temperature_instant,14.14,basic-day,FORECAST,2026-05-10 23:41:59.512551
6,LOC001,34.12345,-78.23456,US,2026-05-16,temperature_instant,13.91,basic-day,FORECAST,2026-05-10 23:41:59.512551
7,LOC001,34.12345,-78.23456,US,2026-05-10,precipitation,0.0,basic-day,FORECAST,2026-05-10 23:41:59.512551
8,LOC001,34.12345,-78.23456,US,2026-05-11,precipitation,9.86,basic-day,FORECAST,2026-05-10 23:41:59.512551
9,LOC001,34.12345,-78.23456,US,2026-05-12,precipitation,0.0,basic-day,FORECAST,2026-05-10 23:41:59.512551


In [11]:
# ============================================================
# Cell 11: Create Location Master DataFrame
# ============================================================

location_master_df = locations_df.copy()

# Standardize data types
location_master_df["latitude"] = pd.to_numeric(location_master_df["latitude"], errors="coerce")
location_master_df["longitude"] = pd.to_numeric(location_master_df["longitude"], errors="coerce")
location_master_df["start_date"] = pd.to_datetime(location_master_df["start_date"]).dt.date
location_master_df["end_date"] = pd.to_datetime(location_master_df["end_date"]).dt.date

# Add useful control flags for future scaling
location_master_df["active_flag"] = True
location_master_df["weather_required_flag"] = True
location_master_df["soil_required_flag"] = True

# Placeholder fields for later GIS enhancement
location_master_df["land_flag"] = None
location_master_df["coastal_belt_flag"] = None

# Add audit timestamp
location_master_df["created_ts"] = pd.Timestamp.now()

print("Location master shape:", location_master_df.shape)
display(location_master_df)

Location master shape: (1, 12)


,location_id,latitude,longitude,country_code,start_date,end_date,active_flag,weather_required_flag,soil_required_flag,land_flag,coastal_belt_flag,created_ts
0,LOC001,34.12345,-78.23456,US,2024-04-10,2024-04-15,True,True,True,None,None,2026-05-10 23:46:40.470960


In [12]:
# ============================================================
# Cell 12: Save Location Master to output folder
# ============================================================

location_master_file = output_folder / "location_master.csv"

location_master_df.to_csv(location_master_file, index=False)

print("Location Master CSV saved successfully.")
print("File path:", location_master_file)
print("File exists:", location_master_file.exists())

display(location_master_df)

Location Master CSV saved successfully.
File path: c:\Users\dubey\Documents\UC_GitLab\use_cases\meteoblue\data\output\location_master.csv
File exists: True


,location_id,latitude,longitude,country_code,start_date,end_date,active_flag,weather_required_flag,soil_required_flag,land_flag,coastal_belt_flag,created_ts
0,LOC001,34.12345,-78.23456,US,2024-04-10,2024-04-15,True,True,True,None,None,2026-05-10 23:46:40.470960


In [13]:
# ============================================================
# Cell 13: Create Extraction Run Summary
# ============================================================

extraction_run_df = pd.DataFrame([
    {
        "extraction_run_id": f"WEATHER_BASIC_DAY_{location_id}_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}",
        "location_id": location_id,
        "extraction_type": "WEATHER",
        "source_api_package": "basic-day",
        "status": "SUCCESS",
        "input_location_count": len(locations_df),
        "output_weather_day_count": weather_daily_df.shape[0],
        "output_weather_fact_count": weather_long_df.shape[0],
        "raw_file_path": str(raw_weather_file),
        "wide_output_file_path": str(weather_wide_file),
        "long_output_file_path": str(weather_long_file),
        "extraction_ts": pd.Timestamp.now()
    }
])

print("Extraction run summary:")
display(extraction_run_df)

Extraction run summary:


,extraction_run_id,location_id,extraction_type,source_api_package,status,input_location_count,output_weather_day_count,output_weather_fact_count,raw_file_path,wide_output_file_path,long_output_file_path,extraction_ts
0,WEATHER_BASIC_DAY_LOC001_20260510_234834,LOC001,WEATHER,basic-day,SUCCESS,1,7,196,c:\Users\dubey\Documents\UC_GitLab\use_cases\m...,c:\Users\dubey\Documents\UC_GitLab\use_cases\m...,c:\Users\dubey\Documents\UC_GitLab\use_cases\m...,2026-05-10 23:48:34.794056


In [14]:
# ============================================================
# Cell 14: Save Extraction Run Summary
# ============================================================

extraction_run_file = output_folder / "extraction_run_summary.csv"

extraction_run_df.to_csv(extraction_run_file, index=False)

print("Extraction run summary saved successfully.")
print("File path:", extraction_run_file)
print("File exists:", extraction_run_file.exists())

display(extraction_run_df)

Extraction run summary saved successfully.
File path: c:\Users\dubey\Documents\UC_GitLab\use_cases\meteoblue\data\output\extraction_run_summary.csv
File exists: True


,extraction_run_id,location_id,extraction_type,source_api_package,status,input_location_count,output_weather_day_count,output_weather_fact_count,raw_file_path,wide_output_file_path,long_output_file_path,extraction_ts
0,WEATHER_BASIC_DAY_LOC001_20260510_234834,LOC001,WEATHER,basic-day,SUCCESS,1,7,196,c:\Users\dubey\Documents\UC_GitLab\use_cases\m...,c:\Users\dubey\Documents\UC_GitLab\use_cases\m...,c:\Users\dubey\Documents\UC_GitLab\use_cases\m...,2026-05-10 23:48:34.794056


In [16]:
# ============================================================
# Cell 15: Read and inspect reference Meteobe request JSON files
# ============================================================

reference_config_folder = (
    PROJECT_ROOT
    / "reference_repo"
    / "meteobe"
    / "src"
    / "meteobe"
    / "config"
)

soil_request_file = reference_config_folder / "soil_request.json"
weather_request_file = reference_config_folder / "weather_request.json"
codes_file = reference_config_folder / "codes.json"

print("Reference config folder exists:", reference_config_folder.exists())
print("Soil request file exists:", soil_request_file.exists())
print("Weather request file exists:", weather_request_file.exists())
print("Codes file exists:", codes_file.exists())

with open(soil_request_file, "r", encoding="utf-8") as file:
    reference_soil_request = json.load(file)

with open(weather_request_file, "r", encoding="utf-8") as file:
    reference_weather_request = json.load(file)

with open(codes_file, "r", encoding="utf-8") as file:
    reference_codes = json.load(file)

print("\nSoil request object type:")
print(type(reference_soil_request))

print("\nWeather request object type:")
print(type(reference_weather_request))

print("\nCodes object type:")
print(type(reference_codes))

# Soil request inspection
if isinstance(reference_soil_request, dict):
    print("\nSoil request top-level keys:")
    print(list(reference_soil_request.keys()))
elif isinstance(reference_soil_request, list):
    print("\nSoil request is a list.")
    print("Number of soil query blocks:", len(reference_soil_request))
    print("First soil query block:")
    print(json.dumps(reference_soil_request[0], indent=2)[:1500])

# Weather request inspection
if isinstance(reference_weather_request, dict):
    print("\nWeather request top-level keys:")
    print(list(reference_weather_request.keys()))
elif isinstance(reference_weather_request, list):
    print("\nWeather request is a list.")
    print("Number of weather query blocks:", len(reference_weather_request))
    print("First weather query block:")
    print(json.dumps(reference_weather_request[0], indent=2)[:1500])

# Codes inspection
if isinstance(reference_codes, dict):
    print("\nCodes top-level keys:")
    print(list(reference_codes.keys())[:20])
elif isinstance(reference_codes, list):
    print("\nCodes is a list.")
    print("Number of code records:", len(reference_codes))
    print("First code record:")
    print(json.dumps(reference_codes[0], indent=2)[:1500])

Reference config folder exists: True
Soil request file exists: True
Weather request file exists: True
Codes file exists: True

Soil request object type:
<class 'list'>

Weather request object type:
<class 'list'>

Codes object type:
<class 'list'>

Soil request is a list.
Number of soil query blocks: 2
First soil query block:
{
  "domain": "SOILGRIDS2",
  "codes": [
    {
      "code": 808,
      "level": "aggregated",
      "startDepth": 0,
      "endDepth": 30
    },
    {
      "code": 809,
      "level": "aggregated",
      "startDepth": 0,
      "endDepth": 30
    },
    {
      "code": 803,
      "level": "aggregated",
      "startDepth": 0,
      "endDepth": 30
    },
    {
      "code": 807,
      "level": "aggregated",
      "startDepth": 0,
      "endDepth": 30
    },
    {
      "code": 811,
      "level": "aggregated",
      "startDepth": 0,
      "endDepth": 30
    },
    {
      "code": 838,
      "level": "aggregated",
      "startDepth": 0,
      "endDepth": 30
    },
 

In [17]:
# ============================================================
# Cell 16: Summarize soil and weather query blocks from reference repo
# ============================================================

print("SOIL QUERY BLOCK SUMMARY")
print("=" * 80)

for i, block in enumerate(reference_soil_request, start=1):
    domain = block.get("domain")
    codes = block.get("codes", [])
    
    print(f"\nSoil block {i}")
    print("Domain:", domain)
    print("Number of requested codes:", len(codes))
    
    depth_summary = []
    for c in codes:
        depth_summary.append({
            "code": c.get("code"),
            "level": c.get("level"),
            "startDepth": c.get("startDepth"),
            "endDepth": c.get("endDepth"),
        })
    
    display(pd.DataFrame(depth_summary))


print("\n\nWEATHER QUERY BLOCK SUMMARY")
print("=" * 80)

for i, block in enumerate(reference_weather_request, start=1):
    domain = block.get("domain")
    time_resolution = block.get("timeResolution")
    codes = block.get("codes", [])
    
    print(f"\nWeather block {i}")
    print("Domain:", domain)
    print("Time resolution:", time_resolution)
    print("Number of requested codes:", len(codes))
    
    weather_summary = []
    for c in codes:
        weather_summary.append({
            "code": c.get("code"),
            "level": c.get("level"),
            "aggregation": c.get("aggregation"),
        })
    
    display(pd.DataFrame(weather_summary).head(30))

SOIL QUERY BLOCK SUMMARY

Soil block 1
Domain: SOILGRIDS2
Number of requested codes: 11


,code,level,startDepth,endDepth
0,808,aggregated,0.0,30.0
1,809,aggregated,0.0,30.0
2,803,aggregated,0.0,30.0
3,807,aggregated,0.0,30.0
4,811,aggregated,0.0,30.0
5,838,aggregated,0.0,30.0
6,837,0-30 cm,NaN,NaN
7,805,aggregated,0.0,30.0
8,804,aggregated,0.0,30.0
9,817,aggregated,0.0,30.0



Soil block 2
Domain: SOILGRIDS2
Number of requested codes: 11


,code,level,startDepth,endDepth
0,808,aggregated,0.0,60.0
1,809,aggregated,0.0,60.0
2,803,aggregated,0.0,60.0
3,807,aggregated,0.0,60.0
4,811,aggregated,0.0,60.0
5,838,aggregated,0.0,60.0
6,837,0-30 cm,NaN,NaN
7,805,aggregated,0.0,60.0
8,804,aggregated,0.0,60.0
9,817,aggregated,0.0,60.0




WEATHER QUERY BLOCK SUMMARY

Weather block 1
Domain: NEMSGLOBAL
Time resolution: daily
Number of requested codes: 21


...

,code,level,aggregation
0,61,sfc,sum



Weather block 4
Domain: ERA5T
Time resolution: daily
Number of requested codes: 4


,code,level,aggregation
0,32,10 m above gnd,max
1,32,10 m above gnd,min
2,32,10 m above gnd,mean
3,735,10 m above gnd,None



Weather block 5
Domain: ERA5
Time resolution: hourly
Number of requested codes: 1


,code,level,aggregation
0,721,sfc,None


In [18]:
# ============================================================
# Cell 17: Create Meteoblue variable code lookup table
# ============================================================

codes_df = pd.DataFrame(reference_codes)

print("Codes lookup shape:", codes_df.shape)
print("Columns:", codes_df.columns.tolist())

display(codes_df.head(20))

Codes lookup shape: (229, 3)
Columns: ['defaultUnit', 'code', 'variable']


,defaultUnit,code,variable
0,Pa,1,Pressure
1,hPa,2,Mean Sea Level Pressure
2,Gpm,7,Geopotential Height
3,m,8,Height/Elevation
4,°C,11,Temperature
5,°C,15,Daily Max Temperature
6,°C,16,Daily Min Temperature
7,°C,17,Dewpoint Temperature
8,m,20,Visibility
9,°,31,Wind Direction


In [19]:
# ============================================================
# Cell 18: Map soil request codes to readable variable names
# ============================================================

soil_request_rows = []

for block_index, block in enumerate(reference_soil_request, start=1):
    domain = block.get("domain")
    codes = block.get("codes", [])

    for code_item in codes:
        soil_request_rows.append({
            "query_block": block_index,
            "domain": domain,
            "variable_code": code_item.get("code"),
            "level": code_item.get("level"),
            "start_depth_cm": code_item.get("startDepth"),
            "end_depth_cm": code_item.get("endDepth"),
        })

soil_request_df = pd.DataFrame(soil_request_rows)

soil_request_mapped_df = soil_request_df.merge(
    codes_df,
    left_on="variable_code",
    right_on="code",
    how="left"
).drop(columns=["code"])

soil_request_mapped_df = soil_request_mapped_df.rename(columns={
    "variable": "variable_name",
    "defaultUnit": "default_unit"
})

print("Mapped soil request shape:", soil_request_mapped_df.shape)
display(soil_request_mapped_df)

Mapped soil request shape: (22, 8)


,query_block,domain,variable_code,level,start_depth_cm,end_depth_cm,default_unit,variable_name
0,1,SOILGRIDS2,808,aggregated,0.0,30.0,kg/m³,Bulk Density
1,1,SOILGRIDS2,809,aggregated,0.0,30.0,cmolc/kg,Cation Exchange Capacity
2,1,SOILGRIDS2,803,aggregated,0.0,30.0,%,Clay Content (0-2 micro meter) mass fraction
3,1,SOILGRIDS2,807,aggregated,0.0,30.0,vol. %,Coarse Fragments volumetric fraction
4,1,SOILGRIDS2,811,aggregated,0.0,30.0,%,Organic Carbon Content (fine earth fraction)
5,1,SOILGRIDS2,838,aggregated,0.0,30.0,kg/m³,Organic Carbon Density
6,1,SOILGRIDS2,837,0-30 cm,NaN,NaN,kg/m²,Organic Carbon Stocks
7,1,SOILGRIDS2,805,aggregated,0.0,30.0,%,Sand content (50-2000 micro meter) mass fraction
8,1,SOILGRIDS2,804,aggregated,0.0,30.0,%,Silt Content (2-50 micro meter) mass fraction
9,1,SOILGRIDS2,817,aggregated,0.0,30.0,g/kg,Total Nitrogen Content


In [20]:
# ============================================================
# Cell 19: Map weather request codes to readable variable names
# ============================================================

weather_request_rows = []

for block_index, block in enumerate(reference_weather_request, start=1):
    domain = block.get("domain")
    time_resolution = block.get("timeResolution")
    codes = block.get("codes", [])

    for code_item in codes:
        weather_request_rows.append({
            "query_block": block_index,
            "domain": domain,
            "time_resolution": time_resolution,
            "variable_code": code_item.get("code"),
            "level": code_item.get("level"),
            "aggregation": code_item.get("aggregation"),
        })

weather_request_df = pd.DataFrame(weather_request_rows)

weather_request_mapped_df = weather_request_df.merge(
    codes_df,
    left_on="variable_code",
    right_on="code",
    how="left"
).drop(columns=["code"])

weather_request_mapped_df = weather_request_mapped_df.rename(columns={
    "variable": "variable_name",
    "defaultUnit": "default_unit"
})

print("Mapped weather request shape:", weather_request_mapped_df.shape)
display(weather_request_mapped_df.head(50))

Mapped weather request shape: (30, 8)


,query_block,domain,time_resolution,variable_code,level,aggregation,default_unit,variable_name
0,1,NEMSGLOBAL,daily,52,2 m above gnd,max,%,Relative Humidity
1,1,NEMSGLOBAL,daily,52,2 m above gnd,min,%,Relative Humidity
2,1,NEMSGLOBAL,daily,52,2 m above gnd,mean,%,Relative Humidity
3,1,NEMSGLOBAL,daily,71,sfc,mean,%,Cloud Cover Total
4,1,NEMSGLOBAL,daily,75,high cld lay,mean,%,Cloud Cover High
5,1,NEMSGLOBAL,daily,74,mid cld lay,mean,%,Cloud Cover Medium
6,1,NEMSGLOBAL,daily,73,low cld lay,mean,%,Cloud Cover Low
7,1,NEMSGLOBAL,daily,191,sfc,sum,min,Sunshine Duration
8,1,NEMSGLOBAL,daily,204,sfc,mean,W/m²,Shortwave Radiation
9,1,NEMSGLOBAL,daily,258,sfc,mean,W/m²,Direct Shortwave Radiation


In [21]:
# ============================================================
# Cell 20: Create proposed Soil Fact schema
# ============================================================

soil_fact_schema = pd.DataFrame([
    {"column_name": "location_id", "data_type": "STRING", "description": "Unique location identifier from input file"},
    {"column_name": "latitude", "data_type": "FLOAT", "description": "Latitude of extraction point"},
    {"column_name": "longitude", "data_type": "FLOAT", "description": "Longitude of extraction point"},
    {"column_name": "domain", "data_type": "STRING", "description": "Meteoblue dataset domain, e.g., SOILGRIDS2"},
    {"column_name": "variable_code", "data_type": "NUMBER", "description": "Meteoblue numeric variable code"},
    {"column_name": "variable_name", "data_type": "STRING", "description": "Readable soil variable name from codes lookup"},
    {"column_name": "default_unit", "data_type": "STRING", "description": "Default unit from Meteoblue codes lookup"},
    {"column_name": "level", "data_type": "STRING", "description": "Soil level/depth description from request"},
    {"column_name": "start_depth_cm", "data_type": "NUMBER", "description": "Start depth in centimeters, where available"},
    {"column_name": "end_depth_cm", "data_type": "NUMBER", "description": "End depth in centimeters, where available"},
    {"column_name": "soil_value", "data_type": "FLOAT", "description": "Extracted soil value"},
    {"column_name": "source_api", "data_type": "STRING", "description": "Source API, e.g., Meteoblue Dataset API"},
    {"column_name": "extraction_run_id", "data_type": "STRING", "description": "Audit key linking to extraction run"},
    {"column_name": "extraction_ts", "data_type": "TIMESTAMP", "description": "Timestamp when extraction was performed"},
])

print("Proposed Soil Fact schema:")
display(soil_fact_schema)

Proposed Soil Fact schema:


,column_name,data_type,description
0,location_id,STRING,Unique location identifier from input file
1,latitude,FLOAT,Latitude of extraction point
2,longitude,FLOAT,Longitude of extraction point
3,domain,STRING,"Meteoblue dataset domain, e.g., SOILGRIDS2"
4,variable_code,NUMBER,Meteoblue numeric variable code
5,variable_name,STRING,Readable soil variable name from codes lookup
6,default_unit,STRING,Default unit from Meteoblue codes lookup
7,level,STRING,Soil level/depth description from request
8,start_depth_cm,NUMBER,"Start depth in centimeters, where available"
9,end_depth_cm,NUMBER,"End depth in centimeters, where available"


In [22]:
# ============================================================
# Cell 21: Create proposed Weather Fact schema
# ============================================================

weather_fact_schema = pd.DataFrame([
    {"column_name": "location_id", "data_type": "STRING", "description": "Unique location identifier from input file"},
    {"column_name": "latitude", "data_type": "FLOAT", "description": "Latitude of extraction point"},
    {"column_name": "longitude", "data_type": "FLOAT", "description": "Longitude of extraction point"},
    {"column_name": "country_code", "data_type": "STRING", "description": "Country code used for domain selection"},
    {"column_name": "weather_date", "data_type": "DATE", "description": "Weather observation or forecast date"},
    {"column_name": "domain", "data_type": "STRING", "description": "Meteoblue dataset domain, e.g., NEMSGLOBAL, CHIRPS2, ERA5T"},
    {"column_name": "time_resolution", "data_type": "STRING", "description": "Time resolution of requested data, e.g., daily or hourly"},
    {"column_name": "variable_code", "data_type": "NUMBER", "description": "Meteoblue numeric variable code"},
    {"column_name": "variable_name", "data_type": "STRING", "description": "Readable weather variable name from codes lookup"},
    {"column_name": "default_unit", "data_type": "STRING", "description": "Default unit from Meteoblue codes lookup"},
    {"column_name": "level", "data_type": "STRING", "description": "Atmospheric/soil/surface level from request"},
    {"column_name": "aggregation", "data_type": "STRING", "description": "Aggregation method, e.g., max, min, mean, sum"},
    {"column_name": "weather_value", "data_type": "FLOAT", "description": "Extracted weather value"},
    {"column_name": "source_api", "data_type": "STRING", "description": "Source API, e.g., Meteoblue Dataset API or basic-day"},
    {"column_name": "extraction_type", "data_type": "STRING", "description": "FORECAST or HISTORICAL"},
    {"column_name": "extraction_run_id", "data_type": "STRING", "description": "Audit key linking to extraction run"},
    {"column_name": "extraction_ts", "data_type": "TIMESTAMP", "description": "Timestamp when extraction was performed"},
])

print("Proposed Weather Fact schema:")
display(weather_fact_schema)

Proposed Weather Fact schema:


,column_name,data_type,description
0,location_id,STRING,Unique location identifier from input file
1,latitude,FLOAT,Latitude of extraction point
2,longitude,FLOAT,Longitude of extraction point
3,country_code,STRING,Country code used for domain selection
4,weather_date,DATE,Weather observation or forecast date
5,domain,STRING,"Meteoblue dataset domain, e.g., NEMSGLOBAL, CH..."
6,time_resolution,STRING,"Time resolution of requested data, e.g., daily..."
7,variable_code,NUMBER,Meteoblue numeric variable code
8,variable_name,STRING,Readable weather variable name from codes lookup
9,default_unit,STRING,Default unit from Meteoblue codes lookup


In [23]:
# ============================================================
# Cell 22: Save Phase 1 reference/model outputs
# ============================================================

# Save request mappings
soil_request_mapping_file = output_folder / "reference_soil_request_mapping.csv"
weather_request_mapping_file = output_folder / "reference_weather_request_mapping.csv"

soil_request_mapped_df.to_csv(soil_request_mapping_file, index=False)
weather_request_mapped_df.to_csv(weather_request_mapping_file, index=False)

# Save proposed schemas
soil_fact_schema_file = output_folder / "proposed_soil_fact_schema.csv"
weather_fact_schema_file = output_folder / "proposed_weather_fact_schema.csv"

soil_fact_schema.to_csv(soil_fact_schema_file, index=False)
weather_fact_schema.to_csv(weather_fact_schema_file, index=False)

print("Phase 1 reference/model outputs saved successfully.")
print()
print("Saved files:")
print(soil_request_mapping_file)
print(weather_request_mapping_file)
print(soil_fact_schema_file)
print(weather_fact_schema_file)

print("\nFile existence check:")
print("Soil request mapping:", soil_request_mapping_file.exists())
print("Weather request mapping:", weather_request_mapping_file.exists())
print("Soil fact schema:", soil_fact_schema_file.exists())
print("Weather fact schema:", weather_fact_schema_file.exists())

Phase 1 reference/model outputs saved successfully.

Saved files:
c:\Users\dubey\Documents\UC_GitLab\use_cases\meteoblue\data\output\reference_soil_request_mapping.csv
c:\Users\dubey\Documents\UC_GitLab\use_cases\meteoblue\data\output\reference_weather_request_mapping.csv
c:\Users\dubey\Documents\UC_GitLab\use_cases\meteoblue\data\output\proposed_soil_fact_schema.csv
c:\Users\dubey\Documents\UC_GitLab\use_cases\meteoblue\data\output\proposed_weather_fact_schema.csv

File existence check:
Soil request mapping: True
Weather request mapping: True
Soil fact schema: True
Weather fact schema: True


In [24]:
# ============================================================
# Cell 23: Install/import Meteoblue Dataset SDK for soil test
# Phase 1.5 — Soil API Feasibility Test
# ============================================================

import sys

# Install into the active notebook kernel
!{sys.executable} -m pip install meteoblue-dataset-sdk

print("meteoblue-dataset-sdk installation/check complete.")


  Attempting uninstall: protobuf

    Found existing installation: protobuf 7.34.0

    Uninstalling protobuf-7.34.0:

      Successfully uninstalled protobuf-7.34.0

   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 3/3 [meteoblue-dataset-sdk]

meteoblue-dataset-sdk installation/check complete.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
# ============================================================
# Cell 23A: Import check for Meteoblue Dataset SDK
# ============================================================

import meteoblue_dataset_sdk

print("Meteoblue Dataset SDK imported successfully.")
print("SDK module:", meteoblue_dataset_sdk)

Meteoblue Dataset SDK imported successfully.
SDK module: <module 'meteoblue_dataset_sdk' from 'c:\\Users\\dubey\\Documents\\UC_GitLab\\.venv_global\\Lib\\site-packages\\meteoblue_dataset_sdk\\__init__.py'>


In [26]:
# ============================================================
# Cell 24: Inspect Meteoblue Dataset SDK structure
# Phase 1.5 — Soil API Feasibility Test
# ============================================================

import inspect
import meteoblue_dataset_sdk

print("SDK module file:")
print(meteoblue_dataset_sdk.__file__)

print("\nAvailable SDK attributes/classes/functions:")
sdk_items = [item for item in dir(meteoblue_dataset_sdk) if not item.startswith("_")]
for item in sdk_items:
    print(item)

print("\nDetailed object types:")
for item in sdk_items:
    obj = getattr(meteoblue_dataset_sdk, item)
    print(f"{item}: {type(obj)}")

SDK module file:
c:\Users\dubey\Documents\UC_GitLab\.venv_global\Lib\site-packages\meteoblue_dataset_sdk\__init__.py

Available SDK attributes/classes/functions:
ApiError
Client
Error
caching
client
protobuf
utils

Detailed object types:
ApiError: <class 'type'>
Client: <class 'type'>
Error: <class 'type'>
caching: <class 'module'>
client: <class 'module'>
protobuf: <class 'module'>
utils: <class 'module'>


In [27]:
# ============================================================
# Cell 25: Inspect Meteoblue Dataset SDK Client
# Phase 1.5 — Soil API Feasibility Test
# ============================================================

from meteoblue_dataset_sdk import Client
import inspect

print("Client class:")
print(Client)

print("\nClient constructor signature:")
print(inspect.signature(Client))

print("\nClient public methods/attributes:")
client_items = [item for item in dir(Client) if not item.startswith("_")]
for item in client_items:
    obj = getattr(Client, item)
    print(f"{item}: {type(obj)}")

print("\nSource/help for Client if available:")
print(inspect.getdoc(Client))

Client class:
<class 'meteoblue_dataset_sdk.client.Client'>

Client constructor signature:
(apikey: str, cache: Optional[meteoblue_dataset_sdk.caching.filecache.FileCache] = None)

Client public methods/attributes:
measurement_query: <class 'function'>
measurement_sync: <class 'function'>
query: <class 'function'>
querySync: <class 'function'>
query_sync: <class 'function'>

Source/help for Client if available:
None


In [28]:
# ============================================================
# Cell 26: Inspect Client query_sync method
# Phase 1.5 — Soil API Feasibility Test
# ============================================================

print("query_sync signature:")
print(inspect.signature(Client.query_sync))

print("\nquery_sync documentation:")
print(inspect.getdoc(Client.query_sync))

print("\nquery signature:")
print(inspect.signature(Client.query))

print("\nquery documentation:")
print(inspect.getdoc(Client.query))

query_sync signature:
(self, params: dict)

query_sync documentation:
Exactly the same as querySync but using underscore in the name.
Keeping querySync for backward compatibility.

query signature:
(self, params: dict)

query documentation:
Query meteoblue dataset api asynchronously, transfer data using protobuf and
return a structured object

:param params:
    query parameters,
    see https://docs.meteoblue.com/en/apis/environmental-data/dataset-api
:return: DatasetApiProtobuf object


In [29]:
# ============================================================
# Cell 27: Build one soil Dataset API request
# Phase 1.5 — Soil API Feasibility Test
# ============================================================

# Use the same location/date values selected from the input CSV
# Soil is mostly static, but the Dataset API request still expects a time interval structure.
soil_time_interval = f"{start_date}T+10:00/{end_date}T+10:00"

soil_params = {
    "units": {
        "temperature": "CELSIUS",
        "velocity": "KILOMETER_PER_HOUR",
        "length": "metric",
        "energy": "watts"
    },
    "geometry": {
        "type": "MultiPoint",
        "coordinates": [[longitude, latitude]],  # Meteoblue expects lon, lat order
        "locationNames": [location_id],
        "mode": "preferLandWithMatchingElevation"
    },
    "format": "json",
    "timeIntervals": [
        soil_time_interval
    ],
    "timeIntervalsAlignment": "none",
    "queries": reference_soil_request
}

print("Soil Dataset API request built successfully.")
print("Location:", location_id)
print("Coordinates:", [[longitude, latitude]])
print("Time interval:", soil_time_interval)
print("Number of soil query blocks:", len(soil_params["queries"]))

print("\nSoil request preview:")
print(json.dumps(soil_params, indent=2)[:3000])

Soil Dataset API request built successfully.
Location: LOC001
Coordinates: [[-78.23456, 34.12345]]
Time interval: 2024-04-10T+10:00/2024-04-15T+10:00
Number of soil query blocks: 2

Soil request preview:
{
  "units": {
    "temperature": "CELSIUS",
    "velocity": "KILOMETER_PER_HOUR",
    "length": "metric",
    "energy": "watts"
  },
  "geometry": {
    "type": "MultiPoint",
    "coordinates": [
      [
        -78.23456,
        34.12345
      ]
    ],
    "locationNames": [
      "LOC001"
    ],
    "mode": "preferLandWithMatchingElevation"
  },
  "format": "json",
  "timeIntervals": [
    "2024-04-10T+10:00/2024-04-15T+10:00"
  ],
  "timeIntervalsAlignment": "none",
  "queries": [
    {
      "domain": "SOILGRIDS2",
      "codes": [
        {
          "code": 808,
          "level": "aggregated",
          "startDepth": 0,
          "endDepth": 30
        },
        {
          "code": 809,
          "level": "aggregated",
          "startDepth": 0,
          "endDepth": 30
     

In [30]:
# ============================================================
# Cell 28: Call Meteoblue Dataset API for soil data
# Phase 1.5 — Soil API Feasibility Test
# ============================================================

from meteoblue_dataset_sdk import Client, ApiError

client = Client(apikey=METEOBLUE_API_KEY)

try:
    soil_response = client.query_sync(soil_params)
    
    print("Soil Dataset API call completed successfully.")
    print("Response object type:", type(soil_response))
    print("Response object:", soil_response)

except ApiError as api_error:
    print("Meteoblue API error occurred.")
    print(type(api_error))
    print(api_error)

except Exception as exc:
    print("General error occurred.")
    print(type(exc))
    print(exc)

Meteoblue API error occurred.
<class 'meteoblue_dataset_sdk.client.ApiError'>
Access to the dataset API is not available for this user
